# 冷轧卷序优化示例

这个示例面向钢铁企业的冷轧计划、工艺和质量人员，说明如何把更泛化的冷轧卷序问题表达成一个清晰、可解释、可求解的优化模型。

这里不绑定某一条实际机组或某个企业的专有规则，而是用一个通用冷轧场景说明：当一批待轧合同卷存在不同入口厚度、目标厚度、宽度、钢种族、硬度族、表面等级、涂油路线和交期窗口时，如何让求解器在巨大排列空间中搜索更平稳、更容易执行的顺序。

## 冷轧业务需求

冷轧排程不是简单地按订单号或交期排序。卷序会影响轧制稳定性、板形控制、辊耗、表面质量和计划兑现。即使不具体到某一条机组，冷轧场景通常也会遇到这些共性需求：

- **厚度和压下率跳变影响轧制稳定**：入口厚度、目标厚度或压下率变化过大，会增加设定调整和张力/轧制力波动。
- **宽度跳跃影响板形和辊形策略**：宽度大幅上跳或频繁宽窄交替，会增加板形控制、边浪和换辊风险。
- **钢种和硬度族切换需要缓冲**：软钢、深冲钢、高强钢等材料混排时，轧制力、延伸率和工艺窗口差异明显。
- **表面质量窗口需要保护**：高表面等级或外观敏感材料通常希望成组生产，减少被普通材料或剧烈过渡打散。
- **涂油和后续路线切换增加操作负担**：不同防锈油、干/湿路线或下游去向频繁切换，会降低计划可执行性。
- **交期压力和工艺平稳性相互冲突**：单纯按交期可能导致规格剧烈跳变，单纯按规格又可能推迟急单。

因此，一个有价值的冷轧排程模型不只是给出“可行顺序”，还应能解释每个顺序为什么更平稳、哪些规则被改善、哪些规则为了交期或分组被有意牺牲。

## 求解器能做到什么

冷轧卷序的排列空间增长很快。几十个合同卷已经足以让人工计划只能依赖经验修补，而难以系统比较所有相邻过渡。

一个合适的求解器通常可以做到：

- 把待轧合同卷顺序建成一个排列变量，而不是手工枚举候选顺序；
- 根据相邻卷的宽度、入口厚度、目标厚度、压下率、钢种族、硬度族、表面等级和涂油路线计算过渡成本；
- 从订单表中的可复现默认顺序出发，搜索更低成本的卷序；
- 在交期、规格平滑、表面质量分组和换辊风险之间做权衡；
- 对交期压力做两层表达：相邻卷交期窗口不要剧烈混排，急单也不应在序列中过晚出现；
- 输出总成本和分项规则成本，让计划员能判断优化方向是否符合现场经验。

换句话说，求解器承担的是“在巨大排列空间中找更好方案”的工作；计划员仍然负责确认规则权重、例外处理和最终执行口径。

## OptAgent 能做得更好的地方

冷轧排程的难点不只在求解，还在建模表达。许多现场规则很难一开始就写成完整线性公式，例如压下率过渡、表面等级保护、高强钢连续性、宽度上跳风险和交期偏离惩罚，往往需要先用业务评分函数快速试验。

OptAgent 在这个示例中强调三点：

- **建模声明更清晰**：用 `sequence_var` 直接声明合同卷排列，用 `external_call` 接入 Python 业务评分函数，不需要先把每条规则硬翻译成复杂数学式。
- **规则解释更自然**：宽度平滑、厚度平滑、压下率平滑、钢种切换、硬度族切换、表面质量块、涂油路线和交期偏离都保留为独立命名项。
- **从明确基线开始优化**：模型使用订单表顺序作为可复现基线，先计算基线成本，再搜索更优卷序，便于对比解释。
- **适合逐步工程化**：示例用内联表格和黑箱评分展示能力，后续可以逐步加入正式数据接口、权重标定、审批报表和生产输出。

这使得 OptAgent 适合用来探索冷轧这类“规则多、解释要求高、需要逐步落地”的排程优化问题。

## 示例如何阅读

推荐入口：

```text
src/cold_rolling_model.ipynb
```

这个 notebook 使用内联的订单表、材料表和产能类型表，不依赖 SQLite，也不指向具体真实机组。它会展示：

1. 如何用订单表、材料表和产能类型表描述通用冷轧业务输入；
2. 如何把多张业务表 join 成模型需要的待排合同卷；
3. 如何把宽度、入口厚度、目标厚度、压下率、钢种族、硬度族、表面等级、涂油路线和交期窗口写成可解释的规则成本；
4. 如何把“交期压力和工艺平稳性冲突”拆成 `due_bucket_spread` 和 `due_position_risk` 两类规则；
5. 如何在“需求声明位置对照”中看到每个业务需求对应的字段和规则；
6. 如何用 `sequence_var` 声明卷序变量；
7. 如何用 `external_call` 接入业务评分函数；
8. 如何运行启发式搜索并查看优化前后的规则成本变化；
9. 如何在最后的完整代码示例中看到一个可直接运行的紧凑版本。